# Detalhes do projeto: 

- Versão Python: 3.14.7

# 2. DESAFIO
Você acaba de ser contratado como Analista de Dados Júnior em uma empresa de varejo que possui um histórico de vendas em formato CSV e precisa de um relatório analítico para a reunião trimestral da diretoria. O time de negócios quer entender:

- Como as vendas se comportam ao longo do tempo (por mês e por trimestre);
- Quais produtos e categorias geram mais receita?
- Quais regiões têm melhor desempenho?
- Quais clientes são mais valiosos (segmentação simples por faixa de gasto)?
- Quantas vendas individuais tiveram receita acima da média geral de receita por venda?


In [1]:
# Agrupando importações necessárias para o projeto 
import os
import csv
import json
import random
import re
from datetime import datetime, timedelta
import funcoes_dataset as fdt

# RF01 – Criar ou Carregar o Dataset de Vendas

Gerando sementes do dataset conforme recomendação do guia para o projeto. (Desenvolvedor(a) em IA para Análise Preditiva [T4])

O arquivo, ou pelo menos o nome do arquivo/formato precisam existir antes de executar o comando.

Dei preferência por modularizar o código, já que não faz parte do escopo principal e não manter o código completo no mesmo arquivo do projeto principal.

In [13]:
# Importando função para gerar as sementes do dataset
from gerar_sementes import gerar_dataset_vendas

# chamando a função
gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42)

Dataset gerado com 200 registros em vendas.csv.


# RF02 – Inspecionar e Descrever os Dados

...

Importando o dataset

In [3]:
carregar_dataset()

NameError: name 'carregar_dataset' is not defined

In [ ]:
# # Colando script para semear o dataset
# def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
#     """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
#     random.seed(seed)
#     produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
#                 "Teclado", "Mouse", "Headset"]
#     categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
#                   "Tablet": "Celulares", "Monitor": "Computadores",
#                   "Teclado": "Perifericos", "Mouse": "Perifericos",
#                   "Headset": "Perifericos"}
#     precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
#               "Monitor": 1200, "Teclado": 250, "Mouse": 120,
#               "Headset": 350}
#     regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
#     data_inicio = datetime(2025, 1, 1)
#     colunas = ["id_venda", "data_venda", "cliente", "produto",
#                "categoria", "regiao", "quantidade", "preco_unitario"]

#     with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
#         escritor = csv.DictWriter(arquivo, fieldnames=colunas)
#         escritor.writeheader()

#         for i in range(n_registros):
#             produto = random.choice(produtos)
#             categoria = categorias[produto]
#             quantidade = random.randint(1, 10)
#             preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
#             data = data_inicio + timedelta(days=random.randint(0, 364))
#             data_txt = data.strftime("%Y-%m-%d")
#             cliente = f"Cliente_{random.randint(1, 50):03d}"

#             # --- sujeira proposital para a etapa de limpeza ---
#             if random.random() < 0.05:
#                 quantidade = ""                       # valor ausente
#             if random.random() < 0.04:
#                 preco = ""                            # valor ausente
#             if random.random() < 0.06:
#                 produto = " " + produto + " "         # espacos extras
#             if random.random() < 0.03:
#                 data_txt = "DATA INVALIDA"             # data invalida
#             if random.random() < 0.10:
#                 cliente = random.choice([              # ruido no nome
#                     cliente.upper().replace("_", "-"),
#                     cliente + "!!",
#                     " " + cliente,
#                     cliente.replace("Cliente_", "cliente#"),
#                 ])

#             escritor.writerow({
#                 "id_venda": i + 1,
#                 "data_venda": data_txt,
#                 "cliente": cliente,
#                 "produto": produto,
#                 "categoria": categoria,
#                 "regiao": random.choice(regioes),
#                 "quantidade": quantidade,
#                 "preco_unitario": preco,
#             })

#     print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")

# # gerar_dataset_vendas()
# gerar_dataset_vendas

In [ ]:
# import csv

# def carregar_dataset(caminho_csv):
#     """Le o CSV e retorna uma lista de dicionarios (um por registro)."""
#     with open(caminho_csv, "r", encoding="utf-8") as arquivo:
#         leitor = csv.DictReader(arquivo)
#         registros = list(leitor)
#     return registros

# def inspecionar_dados(registros):
#     """Exibe as informacoes estruturais da lista de registros."""
#     total = len(registros)
#     colunas = list(registros[0].keys()) if registros else []

#     nulos = {coluna: 0 for coluna in colunas}
#     for linha in registros:
#         for coluna in colunas:
#             if linha.get(coluna, "").strip() == "":
#                 nulos[coluna] += 1

#     print("\n=== INSPECAO INICIAL DO DATASET ===")
#     print(f"Total de registros: {total}")
#     print(f"\nColunas: {colunas}")
#     print(f"\nValores ausentes por coluna:\n{nulos}")
#     print("\nPrimeiros registros:")
#     for linha in registros[:5]:
#         print(linha)
#     return registros

# RF03 – Limpar e Tratar os Dados (datetime e regex)

Implemente uma função de limpeza que trate, no mínimo:
- Espaços extras em campos de texto, usando .strip();
- Datas inválidas: converter data_venda com datetime.strptime(...) dentro de um try/except e remover os registros que não puderam ser convertidos;
- Valores ausentes nos campos críticos quantidade e preco_unitario, descartando o registro quando algum deles estiver vazio;
- Tipos de dados: garantir int para quantidade e float para preco_unitario;
Ruído nos nomes de cliente, usando o módulo re: remover caracteres não alfanuméricos, padronizar o formato para - Cliente_NNN e sinalizar os registros fora do padrão;
- Relatório de limpeza: exibir no console quantos registros entraram, quantos foram removidos (por motivo) e quantos permaneceram.

In [ ]:
# conversao de data com tratamento de erro
try:
    data = datetime.strptime(texto_data, "%Y-%m-%d")
except ValueError:
    data = None                       # marca como invalida

# limpeza de texto com expressao regular
nome_limpo = re.sub(r"[^A-Za-z0-9_]", "", nome.strip())

# validacao de formato com padrao compilado
padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

NameError: name 'texto_data' is not defined